# CDCN CVPR 2020 · Kiểm tra MTCNN và PRNet trên OULU-NPU
Dùng runtime **CPU** và chạy các cell theo thứ tự. Mẫu nhỏ lấy 3 video và
9 khung ở mỗi tập Train và Dev của Protocol 1. Mỗi tập có video thật,
ảnh in và phát lại. Kết quả là ảnh scene, bbox, depth JPEG và JSON kiểm tra.
Notebook dành cho bước chuẩn bị dữ liệu trước huấn luyện.

In [ ]:
#@title 1. Kết nối Drive và chọn thư mục OULU-NPU
from pathlib import Path
import os
import shutil
import sys
import time
from google.colab import drive

drive.mount("/content/drive")
DATA = Path("/content/drive/MyDrive/face-pad/datasets/OULU-NPU")
WORK = Path("/content/cdcn-prnet-smoke")
RUN = Path("/content/drive/MyDrive/face-pad/cdcn-prnet-smoke-p1") / str(time.time_ns())
for name, size in {"Protocols.tar": 417792, "Train_files.tar": 37753662976,
                   "Dev_files.tar": 28151992320}.items():
    source = DATA / name
    if not source.is_file() or source.stat().st_size != size:
        raise ValueError(f"Kiểm tra đường dẫn và dung lượng: {source}")
RUN.mkdir(parents=True)
print("Thư mục kết quả:", RUN)

In [ ]:
#@title 2. Nạp code và cài môi trường tách riêng
BUNDLE_SHA256 = 'ec8818a13bbe037528568762e9261091d7a55226b1680c3769034dd68c010696'
BUNDLE = (
    'H4sIAAAAAAAC/+08a0/jyJbzOdL9D7WeD+t0OyYPoGk0uRID6Z7e6QcLzOxeMciq2GXixrG9tgNkctnfvudR5UcSYHr3aqSrxVKD'
    'XY9Tp877nCq6yP2dQKkslL7yMhnseF6URKXnudnyu3/Q04dnf3eXfsOz/rvff/Omfsf2wWg4Gnwn+t/9Cc+iKGUOy3/3//OxLOsE'
    '2P8O2C9Oj05Ergolc38mMunfyGvlwoC/dP7S8bxblRdRmnieGAur7w7cPnR89/L8cz/Fuv5neXoNQlD8efq/tzcYrev/aLT/ov9/'
    'kv6fqjxKg8gXhvUiTHORpKWapumN+JpOCyGTQMRpmhVoDzphns6F54WLcpErMAjRPEvzEgbBJFmCkSg6PCaT5SyOpmbAKXx29Hux'
    'mMJ6PixnWspZrmQQJddVQzRXnU7HjyXgdKqRO+wIeAIVCuOp7ELFoSNiOVWxI0pAIR5/ThPliAX0j62oVPPCckSUlCq/hc5Rv8tQ'
    '8MHJrp5L7wRAvyMA/Wpmg/VrrsSL1MDbcANAAyb011crSm+RBbLETmpCISxVAJ+4aXeeAiXTJPLtbntqUaYZDjKkcie3KilhVEUV'
    'hqtpgus39hqF1CL+ZVyjV/euY42/NjvbyG9gW+ExB16B/9CIwP7KJiZJevfUXlUss4LIgQN7LRpVgzSaFcpVh58ukhJ6QkusaIvO'
    'w86q5i18ilXF3gcLyVL3iqhA2RcoQYBHoQQIfIVXKdH7WU2K0tbEGJrzRZIATyzSlgZE/OyLHxjhHxo9bdpHQazaO27QujWS0bBh'
    'f38Xk4sj8d8ru7Fcj9kuXlVk3KGWQ7cfPhS0XVrqhzXBfiWGrVUqXgANaKkkFYm60+RFG7FCOBpszbxcgVlIYMrlqlauhyugOVLq'
    'YUXzHwCeQW+lXxjSCnb3YNWC5M0gICmnSpa0yYYQ3c0i2AYyq9IN905GPK7aV7dN5SyHDh5hRLRiXNcRYbwoZuOLfKEasux5CoFp'
    'U9MAuA2YFtN1YC0lYv1tKfIFvdkw+VqVYxpV7xw0WSpQlMdhsXbYG1zAIa2N3DdMprr3vXKZKX4DWi3gtcwhFJhC9LduJIm+hSrt'
    '7Qh8TaOk0bWNNKEEhgUkgWZprWIhmPJitkm1TgfRBv54cXp9rQLbT+dz0CdHvKosPvQ4wr8LtNFXyS29afTBX71L8zuZByC482mU'
    'gMD5IDiBSBdltgAvkwIXlPBVzJqaYQic3ypqTfPoOkpAOVSepzk5PwQKSwLz0J3Z8No1bW4mc5AUd34TRLnNHwXtBCkcgTKnNw0W'
    '6r0ApMuizG3gfJfUCl7An5juq05NztD6mF4fihWs9WBtytddVM4IjzRTiW3dW0gNP0X5GluLMuwdWF0hC9y5UzlUW5Oxco/7TfdI'
    'IGtf7Z4S6IoLSHb4x1SHf2jpAwA/bk75cDpxtlmWpmspA6Bwc9b5xcmXXy5AHNV9aUi4theHuVKMrVxlMUgt7w4htHW+zJeHGwgg'
    'pWMQByQ1TnEZ88OtmEKHe5dDJGHjlO6jY4gj9vZ+ZiHOx60EY8vabiFqNxagPyDcyKy1h4AGqawUP8pCTegV4q5N5EHVCECWxrHd'
    'Ref2ecPp1wjCQBCCOUg87HQTo610rGYSjujSkf97m7M1wg0eX/DgyX0W5WD+Hwd9EyH2zyy92Z/LqFBNX40kbS9DQ8QZeCTAZYLi'
    'BEq2YpeFOotBGekAcWOFPx9c8W4BxoLth9HG7j9nLlz4eZSVxY4f+ImX5YkqvWKe3qh/YPr3TP43fDPq72/mf6OX/O9Pyv/OlA8Z'
    'W5kv/FIcnxx/7hXlEuKqTxfHnz/vnJ59ViWYSPSUkIapkrNDKaYQxQWgHCQtEAiUfzgzNG35NXhI0E/97Re35nUmC0wbzSf/2tYA'
    'oUUpITyWpudrkSbmfd7INtPi0YQUPCHgepoW0X0rP50tyih+PFstlnXiKvMQApvqU82z1jemseYdQp6aBP7t0Lwmi3m2RO+VZIxp'
    '4UfZ0i0A4QiTEh717zMwPGSldILdLN24Vf5udqe/Ox3g4uTCO/7y6dOHC0xgQn8wDNWeGoSD3f3hNFByb7jbfzsdDkZB3w8H/oEf'
    '7vt7U6tzMjm9+Kkx098PBv1wqKb7cvp2fyino8FueDBQclfKfl+Gg+DtKJhOYeZ/TD68/+nCO//pCOftvQ1Go4Np2PeHg+mgP1UH'
    'g1E4eBMe7PZ3hz50yr1pOBocqL2Dg4O9vTAcTod+f3fgD/em07cDaXU+fD6Z/KcB508PDgZvh6EfvtndnR6oIaCl5L4c7h8M/OBt'
    'Pxz0h3IYvH3zRk4HEMyO+sP9aRDu+urNwR5MsTrnn778PPFOj45/Pno/Ocd0isPFUiVFmodxejceD93Bvju0HN0T9m5ULgvd3m+0'
    'F3E0H48HVBTVrfPSTxJs69dtxGRsG+67u6YRGH0Tlb1oDiHyeNx3h7v1BBICAvzGHZhGjMD82162LGdp0oMcAZKwArDadQe42sE+'
    'DDSRczGTNsq8jujIk+G3jhDzKYdMoP1Kzg/Xkwethy5KsxdE16qAiJ6GOoDbTA739q2uO1P3us+sGqcy8ObgKuPCJp/iAESwFrnK'
    'NSJp4UK0GOVpcmldvPN+OZ94Hyfvj47/5v08OTs6t66QyQNrY+zxLydH3q8fzj/8+HHinUx+/XA80YN5bKWChokuhKqwYfd2gPss'
    'QxpFqkMMMhPI2HGe9L24eCeQw/9aCJ4cTaM4Kpe0LZVDVJApH+wg5QeQR+hKOXT7ZOJcAgP2wQUSLIA3lyAiHomOG6tr6S89+I7l'
    'UuUF4V6bM37zeJ5dTytyf3Nq9xvWcZM0n8s4+p0wfHLVSgieXn4NIs3q6kSzBPqAkxgzWe1A3Ua+GlvHp78c9g3aILJ4mnCdy2yG'
    'iXDovsdXu5HJtMa4svBAtuQiBjmrJfV78RG4QrzIogSTO8jfgojWJyAS2DnoRUnv4t1wjZ+8b7dOWUExKOAGiRU7wqogQTBkddbD'
    'ctQiTHw9zFDsjTzL1TmJ3Qo3rQ0BZbH8LdkmuUmZA3uY3jTOj39LSHi3DAtzOVd3aX4jagfrFT4ouuU8g8O6ktTYhB6atrW1deMT'
    'y9S7R5vncaRc7IApFa0GaGlWDAzjxuwn3U8kkRfwbtfMaMxoyX494IqYqL9ag4lpUQL5fWn3MVfN2T51a5gbOY66Vz4mvBkYQRuZ'
    'r+ehVYXtYDdWLWph8Tx88bx2PkKkk1lUOeezz61+MpdYVDj7bEcg6qCY43cyLhSBDqP78TZsQyxOxGsYVzvN0szudzf6NMmou0E2'
    'h1JDHs7meisjAkjiZh65K80KrTfGxKPqYOxU7DSGVirUImetRNNlqQq7u522DHmNsNpHGWvjMAWrwdcqAdtVgt+qkTDlpDrgtUlr'
    'nMegaEOTX09hfxCtuf5teZzGmCTyPGw7/vLxy5n34/uz4dn7Hxk3DMkKLKFrsC6/eNRuAzgeBhkpVi+pteF8KSX9FatxnJBaZEZF'
    'EAU0nGGBYQup5EHz7h2xdMSdI5ARc3lvE0xH3KjlOJbzaSAFHoMc0s9La5reW1eXwyvxqtUwuurqV+YrxMQUsyNMJIurY2DcgQOR'
    'qivzXC7tS1j9XrzG5QGJJbzNrrSM3oKmRUwLhnANCaZptKsFjOPIGoJkm2FOY2osk2Au85vmXAf540I4goXMqPikgndRXKq8UVbR'
    'smLXZOo6vJwWCT9PM49SmsIGAjhCgyt8GSuIv0ZaElp0hoHUOFPR9QxinLsoIPxp7uXh8ErHEz9DkE4eigMIiCvAWubkN0Wao55J'
    'EALoLMGB9LI0SkpwZvdgDggjmUegglwx5Do0YKHflhi4Iu3px123C8o3dISNTKAfM25iPGUceog7/iZEcXcgA3c8y3zOqhmxCkst'
    'UGAuMaFyAU8QSYOI6Gmomt85UgJnRIlN5Ng+63V7Fh9qPbrK0qwy0+OnaVmCNeVlDPW3zXvdngf6Rjv661gjCr4G1/7rWIN8Sgkh'
    'q4OYgWgUkOoh/62WfIFbBJcHEB0ND+0ZteGyDi9aheYqRpNwGwUqxTA5LVM/xYPHLI5YmJRXyHkG8kd+oBnC64yTo/h6qpUfcjyP'
    'NziiW9XcDp636WYXApZc+iWCsEPrVM8vdsybN9hZERoPbnlfUiwjA7sLVgz9n80hTq+Irs320zvU8EubCpxYO4ONJ2AgXbCyUWZ3'
    'ua6tC83YQfXt4pYAg6oDei4tiNXRAscjq3C4gWB06UuiBJGMICqJN1UgDVI7OljURX5WLiyu0xCmNVb+8SyZdYmmeXoaoz9wxAAw'
    't3cdMexeIQ4NLvBJhR7E6PizVBu3S5vAOSZmdYT00VLyxtt9VHBuL04lFZqAnbgEqOOuXoUlBBZZPXRM4Xr7akRVxqlm/RbKI7/a'
    'IUMYqTigjSAL0Scw5viGM5iZyCHb8iyNWKOwajOAy/6Vo2FdDupX8Ct4QPoYjbZWrVmOcNqgiwfWtt4iNWwvFW/obGgxkHlUgHXw'
    'Z4dihRt5sDZLxUzkS+zG8JEX2xg1BXm96TRPRrdVk1sYwNoFmHVTJwPFKxfgh7V+iRXRhA6kiSjwxms/tE0L46dNh5pH2nBQqGQI'
    'WdkODq+13zT09WWmg5hfceaxzLBGZ1fxljbFzfDXHKSjTMB09MA2RTxHp97p2ZdT793Z0aeJd/zll88XjaCUCu048wcx6D9LIJ+K'
    'gxTV4MmDEhRaMQSgE4WI61xDAXbEgq8IXF6hSDXr/9+LT8huzgoXU7DA4kTd1u43Sed0qIe2JVCueEcrfjhBwwk5faJ6Uwmw3ZYO'
    'FXFaknaCjxo54s2aEBqYIJ7gdmjwK2EzHXZ2gBBr6QBApI16ENWhSsrkWtkaiEOOzUB8LfZgSYb0GoR/u740gSFhtqsIJotRslCb'
    'h02Sz3Zb7D39cs4sPndq+D1AYWN6eqNHoIgBKHYY2/BEXqc36HnltNguVfWyXVjNLNxFN913975xY4+eWVGYV4ecGynBK609201N'
    'GuFO16NGmsqh6KPnX3Yt/U6joNv9xn3hXR5EnldDC9lYHINPJDGPCOigHUZAxL4AXT74g+bT4koXKBnkGoBnr4DASgkCoSk3l5m1'
    'XRqoIKICb16ZnWjObbblfs2uLU38J4TkKTS/nGscQS9izosYukbs304n77dgBtiMhhqfXBXR78pm1HRo08Cak7sPn84mRyfe+7Oj'
    'v50fH32cdC+B9WBw7BF46NGw+yj6tJIrk6XdRU403RfFJTgG+JGVmU1Du98qAE0jp09mGvZO1ypzhS+FmCowOErQlvG2SWfzAF7N'
    '6coUG1z8fPBWRvcO+6PgwdqchPLg6fRfX6fYYSeERQDKuQsLXiuY8A5LMHSaTaKwjU2Pg4XOx4ES9wcn28HW+H7jTY1tuP0fQGgR'
    'YcnjawXohmv0utqUdP+ICoQNHSBgEExUkB6sJ3ZAo3XppSH5bpnqckz3KRJiJuIVizCM7kGlA4mJAkOkiqj1W2LxpSAAifvDpOge'
    'QvvXArs2QaPXcmUQ2JXF3zzQB7/vygyynsBeWRQBwTjrsJYFPJhAQYE2HQxZrHnQoKOkp6+iWGZ1mGBesWrJzhgajY9+Bow+S1/M'
    'vdEQZmEIxSYBmoCwABJCwSjx+MKX97vK0wpFtNWDh+4z4aeODJEmna21QHbEsYJYpjqamUu8pXWo63Z5QUU+cxTrHuXXC7DS5Sn1'
    '2IHi6wEQl449L0h9U3vjmcgsT+opttXr4Uks2HV0N2M8S31yMFcQ/+hoTMR6nIhZGO8STlZRglXzwG1j40zF2diCVDkVOmOj2Frs'
    '1pnX0HpyEeJZEyWsA/7XAi+nNDT5kcmm/Pm/nc+G7tnZMKWg4wcCQr8QTGFXJQ68SFagPkpz3YezL2omqlftjZuMDE+xUy0WWRYv'
    'hQINBwFhxqIXM1yrinsmd26kpjdqiYhnqSOq+AHDZtsy5CVE9Llg82QaU2+rQUYaZ74d0TyKbnpLvIwWYZrSuFnkz5R/o48Z7Evr'
    'OsJ1rd6xxQVmRA9VMFe3Pdo59v4Ejt66atw465riQzuzodVAQ83uns1wVkCSB0EoAUZivihKcMhiZQA0bbSh6SXMwUSUl+PKj9LF'
    'tZp+6A5PgDc78E5M2hnu7Xv4Dzx96Cf4xtNIHnp0v6aXhvSij1cjPiXmYZRqN07uMe+pOtcsfgSMube6NKU6nH+qgqZDE75SQuTg'
    'WidM6wGmIojCsD7YpF0yB5/2sVwdq4tj1cVGi44GuXiOIRctbDUkhyN8vNHfOLBuSmdLAisFqxWpcbWe9MoEupT+6Dudemg7P26p'
    '4R9PkfkSWg20KTmP5jM0cks+o6MQzhEoUH0GEYs5x34Hs4JEcFG0nQWshzUNRqK88noZX7deSwH4iqTFRuHDp6P3eEPj/NxyeHfY'
    'ydOxVoxOFFsgiqbk2EvSBN0oD+k+cv+ase/UVQS8lLzIuYoH2n/Jta1n/iCDCgIUkKJpsy7QlaMNOVG3TQGrSnftYm9toIEeVfnV'
    'hSWtqoZDYxrlx3oHuoprAuQmMAyDqa7kYW2XIa4RtxpyiH/Qc0M7Ofr1g5ir+RSPmWE/q+YKuvaynZhtfatAi4ujM9I44gTyuVWw'
    'boLfLFo74rfNJF5fqXIvFJ6eynx5Am4RT9YwvcLTaWhfk16FoQ5W3sZUHWLSbxZheN+47Y2ieVXa4bpG65qYzfOIOo/ngDwmKqjS'
    'TnkgF1LJhLpxegdBFtlPy5W3kVUPwKwm4XpTtZFvzA/xOkpsbqkjffAgiKDjj43hfFliywEBb4KvCaWL3FcOg9ZXiO70FSKIFR+5'
    'O8036UBHsyUCTKdfbQMIJzliV7wSg/7Q/NpST5jijXxpziIb5U/CxGlWb4l2V201WquHbk8rYM+YVjTX6m6n0rpFozShcPHuYQw5'
    'ENNFPnb5/ivsPVn/25+mRDYwQP43ETp8NOnQUPUldUTFDRbzrGju5/HUiw7GkvbeUSxHj2CpTebWVKxigmMoA40b0LdkNyxViySO'
    'kpttl7mNHoBDnqe3yq4W2hy7YepWiIA2AaJH6FTwug87ze4HY7X/Llbrtr+n3cKhO8C/eXr6/r42As/obzuna1FXS2SLuvSKVyPQ'
    'I8N3db5Q23DrgcSIT8lAgIoUUW7s16QhdG2N/F4Bgag53hmPre5l/0q7OOUjiPZlySt9MEH3WCD2r9nMHtUyO8AUWr9iXk5Kj3vQ'
    'MS60cWDp6buEh43Qcz2/NkMp6qwnVIGngym3Pr6sejF2fcLZdjcXqR0uZuNrPnhjuKGhpU+WDrdcTXb1dkle+VTNnMOZ6Q8bgAuM'
    '3wKEuy6C+LeBLISccq0bIzpnwrrDHLwk2aR2WaZhGpiDGEQFEFePh/UpueHaE9F8aJ3TiVaU4GWeWJWw+ZWZZ+JSHc/hrXKPgzoj'
    'SRTUkRqCyHRrU7H+518R/tEa0gv/3wHwk56HUux5FqPG5YyX/4bg5Xl5Xp6X5+V5eV6el+fleXlenpfn2ed/AK0Lj7IAUAAA'
)
import base64
import hashlib
import io
import tarfile
bundle = base64.b64decode(BUNDLE)
if hashlib.sha256(bundle).hexdigest() != BUNDLE_SHA256:
    raise ValueError("Code đi kèm không khớp SHA-256")
WORK.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(bundle), mode="r:gz") as archive:
    for member in archive:
        dest = (WORK / member.name).resolve()
        if not member.isfile() or not dest.is_relative_to(WORK):
            raise ValueError(f"Invalid bundle member: {member.name}")
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(archive.extractfile(member).read())
(RUN / f"source-{BUNDLE_SHA256[:12]}.tar.gz").write_bytes(bundle)
sys.path.insert(0, str(WORK / "src"))
from deepface_pad.progress import Progress, run_logged

def run(label, *args):
    run_logged(args, label=label, log=RUN / f"{label}-{time.time_ns()}.log",
               env={**os.environ, "PYTHONPATH": str(WORK / "src"), "TF_ENABLE_ONEDNN_OPTS": "0"})

run("install-uv", sys.executable, "-m", "pip", "install", "uv")
VENV = WORK / "venv"
PY = VENV / "bin/python"
if not PY.is_file():
    run("create-venv", sys.executable, "-m", "uv", "venv", "--python", "3.12.10", VENV)
run("install-prnet", sys.executable, "-m", "uv", "pip", "install", "--python", PY,
    *('tensorflow==2.16.2', 'tf-keras==2.16.0', 'tf-slim==1.1.0', 'mtcnn==1.0.0', 'numpy==1.26.4', 'scikit-image==0.24.0', 'scipy==1.17.1', 'opencv-python-headless==4.11.0.86'))

In [ ]:
#@title 3. Tải code và trọng số PRNet công khai
PRNET = WORK / "PRNet"
RENDERER = WORK / "PRNet-Depth-Generation"
for path, repo, commit in [
    (PRNET, "https://github.com/yfeng95/PRNet.git", "fc12fe5e1f1462bdea52409b213d0cf1c8cf6c5b"),
    (RENDERER, "https://github.com/clks-wzz/PRNet-Depth-Generation.git", "c6d10f2eb6ab962ab314f81ea4aa00af1d93dbbb"),
]:
    if not path.exists():
        run(f"clone-{path.name}", "git", "clone", repo, path)
    run(f"checkout-{path.name}", "git", "-C", path, "checkout", "--detach", commit)
WEIGHT = PRNET / "Data/net-data/256_256_resfcn256_weight.data-00000-of-00001"
def weight_matches(path):
    if not path.is_file():
        return False
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest() == "59d338bf0c21b10be813f17f84042c9d3a5bf318e588855ff2b2c041c25bb91a"

if not weight_matches(WEIGHT):
    partial = WEIGHT.with_suffix(".part")
    run("download-prnet", "curl", "-LfsS", "--retry", "2", "-o", partial,
        "https://drive.usercontent.google.com/download?id=1UoE-XuW1SDLUjZmJPkIZ1MLxvQFgmTFH&export=download&confirm=t")
    if not weight_matches(partial):
        raise ValueError("Tệp tải về không khớp SHA-256 của checkpoint PRNet")
    partial.replace(WEIGHT)
run("verify-prnet", PY, "-u", WORK / "scripts/cdcn_prnet_smoke.py",
    "--image", PRNET / "TestImages/0.jpg", "--prnet", PRNET, "--renderer", RENDERER,
    "--output", WORK / f"public-{time.time_ns()}")

In [ ]:
#@title 4. Dựng dữ liệu mẫu Train/Dev và lưu lên Drive
OUT = WORK / f"oulu-{time.time_ns()}"
WIDE_SAMPLE = False  # True để thêm phone 4/session 2, đọc sâu hơn trong TAR
run("oulu-prnet-smoke", PY, "-u", WORK / "scripts/cdcn_prnet_smoke.py",
    "--data", DATA, "--prnet", PRNET, "--renderer", RENDERER, "--output", OUT,
    *(["--wide-sample"] if WIDE_SAMPLE else []))
with Progress("Save smoke assets to Drive"):
    local_zip = Path(shutil.make_archive(str(OUT), "zip", OUT))
    temporary = RUN / (local_zip.name + ".part")
    shutil.copyfile(local_zip, temporary)
    temporary.replace(RUN / local_zip.name)
    shutil.copyfile(OUT / "smoke_summary.json", RUN / "smoke_summary.json")
print("CDCN_PRNET_SMOKE_READY", RUN)